In [21]:
import numpy as np
import re

In [22]:
class word2vecDataset:
    def __init__(self, window_size=2, k=5):

        if not isinstance(window_size,int) or window_size<1: 
            raise ValueError("Window size must be a positive integer")

        if not isinstance(k,int) or k<1:
            raise ValueError("k must be a positive integer")
        
        self.window_size=window_size
        self.k=k

        self.word2idx={}
        self.idx2word={}
        self.vocab_size=0
        self.idx_corpus=[]
        self.prob_dist=None
        self.training_pairs=[]
        
    # step 1: text processing and vocabulary mapping
    def preprocess(self,corpus):
        # cleaning the corpus to remove all punctuations using regular expressions
        cleaned_corpus=[re.sub(r'[^a-zA-Z\s]',' ',sentence) for sentence in corpus]
        
        # produces list all words of the cleaned corpus in the original order as the corpus and converts them to lowercase
        vocab=" ".join(cleaned_corpus).lower().split() 
        
        uniq_vocab=list(set(vocab)) # extract the unique words only
        self.vocab_size=len(uniq_vocab)

        if self.vocab_size==0:
            raise ValueError("Corpus contains no valid words")
    
        # create two dictionaries: word2idx which maps unique words to indices and idx2word for the reverse mapping
        self.word2idx={word:idx for idx,word in enumerate(uniq_vocab)}
        self.idx2word={idx:word for idx,word in enumerate(uniq_vocab)}
    
        # now convert text corpus to integer IDs
        self.idx_corpus=[[self.word2idx[word] for word in sentence.lower().split()] for sentence in cleaned_corpus]
        
    
    # step 2: generating all pairs of words (centre_word,context_word) as training examples
    def generate_training_pair(self):
        # consider a window size of say, 2 to look atmost 2 words to the left and right of the centre word to capture the true relationship
        self.training_pairs=[]
        
        for sentence in self.idx_corpus:
            sent_len=len(sentence)
            
            for i in range(sent_len):
                start=max(0,i-self.window_size)
                end=min(sent_len,i+self.window_size+1)
    
                for j in range(start,end,1):
                    # capture the pair as (centre_word_idx,context_word_idx)
                    if j!=i: self.training_pairs.append((sentence[i],sentence[j]))
        
    
    # step 3: building the negative sampling distribution
    def prob_distribution(self):
    
        # counting frequencies of all words across the entire text corpus
        counts=np.zeros(self.vocab_size)
        for idx_sentence in self.idx_corpus:
            for word_idx in idx_sentence: counts[word_idx]+=1
    
        # applying the exponential scaling factor to the counts
        scaling_factor=0.75
        scaled_counts=np.power(counts,scaling_factor)
    
        # finding the probability distribution from the scaled counts
        self.prob_dist=scaled_counts/np.sum(scaled_counts)

    
    # this function generates k random noise samples for the centre word which are different from the exclude_idx word
    def k_negative_samples(self, exclude_idx):
        if self.prob_dist is None:
            raise ValueError("No probability distribution created yet, first call prob_distribution().")

        if exclude_idx<0 or exclude_idx>=self.vocab_size:
            raise ValueError("exclude_idx is outside the range of the vocabulary.")

        # Create a copy of the probability distribution created from the prob_distribution() function
        p=self.prob_dist.copy()

        # since the exclude_idx cannot be sampled as negative, assign zero probability to it
        p[exclude_idx]=0.0

        total=p.sum()
        if total<=0:
            raise ValueError("No eligible negative words to sample.")

        # normalize the remaining probabilities by dividing by the sum of all remaining probabilities
        p/=total

        return np.random.choice(
            self.vocab_size,
            size=self.k,
            replace=True,
            p=p
        )

In [23]:
import torch
from torch import nn
import torch.nn.functional as F

class SkipGramNS(nn.Module):
    # step 4: generate the vector embeddings for all words in the text corpus in the constructor for the ANN
    def __init__(self,vocab_size,embedding_dim=16):
        super().__init__()

        self.device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

        if vocab_size<2:
            raise ValueError("The text corpus must have atleast two unique words, currently it does not.")

        if not isinstance(embedding_dim,int) or embedding_dim<1: 
            raise ValueError("Embedding dimension must be a positive integer.")

        # the table for vector embeddings has dimensions (n,d), where n is the number of words in the vocabulary and d is the dimension of the vector embedding for each word
        self.W_centre=nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )
        
        self.W_context=nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )

        with torch.no_grad():
            self.W_centre.weight.normal_(mean=0.0, std=0.01)       
            self.W_context.weight.normal_(mean=0.0, std=0.01)

    def forward(self, centre_id, context_id, negative_ids):
        centre=self.W_centre(centre_id) # dimensions:(,embedding_dim)
        context=self.W_context(context_id) # dimensions:(,embedding_dim)
        negatives=self.W_context(negative_ids) # dimensions: (k,embedding_dim)

        positive_score=torch.dot(centre,context)
        negative_score=negatives@centre

        positive_loss=-F.logsigmoid(positive_score)
        negative_loss=-F.logsigmoid(-negative_score).sum()

        return positive_loss+negative_loss

    # here, data parameter represents the entire word2vec class instance
    def fit(self,data,epochs=50,learning_rate=0.01):
        if not data.training_pairs:
            raise ValueError("Generate training pairs first.")

        if data.prob_dist is None:
            raise ValueError("Create the probability distribution first.")

        if epochs<1 or learning_rate<=0:
            raise ValueError("Enter valid positive values for learning rate and number of epochs.")

        optimizer=torch.optim.Adam(
            self.parameters(),
            lr=learning_rate
        )

        history=[]
        self.train()

        for epoch in range(epochs):
            total_loss=0.0

            # shuffle the indices of training pairs visiting order at every epoch
            order=np.random.permutation(len(data.training_pairs))

            for pair_idx in order:
                # extract training pairs one by one from data.training pairs
                centre_idx,context_idx=data.training_pairs[pair_idx]

                negative_indices=data.k_negative_samples(context_idx)

                # convert everything to PyTorch tensors for compatibility with PyTorch embedded layers
                centre_id=torch.tensor(
                    int(centre_idx), dtype=torch.long, device=self.device
                )

                context_id=torch.tensor(
                    int(context_idx), dtype=torch.long, device=self.device
                )

                negative_ids=torch.tensor(
                    negative_indices, dtype=torch.long, device=self.device
                )

                # clear the gradients from the previous step so that they do not interfere in training of the current batch
                optimizer.zero_grad()

                # calculate the loss
                loss=self.forward(centre_id, context_id, negative_ids)
                # since we have passed PyTorch tensors in the forward() function as input, the loss returned will also by a PyTorch tensor, hence we need to extract the num

                # calculate the gradients
                loss.backward()

                # update the word embeddings
                optimizer.step()

                total_loss+=loss.item()
                
            average_loss=total_loss/len(data.training_pairs)
            history.append(average_loss)

            print(f"Epoch: {epoch+1} : Loss: {average_loss:.4f}")

        return history

In [24]:
def similarity(data, network, word1, word2):
    word1_l,word2_l=word1.lower(),word2.lower()

    if word1_l not in data.word2idx or word2_l not in data.word2idx: 
        raise ValueError("Both words are not present in the vocabulary.")

    word1_idx,word2_idx=data.word2idx[word1_l],data.word2idx[word2_l]
    
    def cosine_similarity(v1,v2):
        dot_product=torch.dot(v1,v2)
        norm1=torch.linalg.vector_norm(v1)
        norm2=torch.linalg.vector_norm(v2)
    
        return dot_product/(norm1*norm2).clamp_min(1e-12)

    # consider centre of word1 and context of word2
    w1_centre=network.W_centre.weight[word1_idx]
    w2_context=network.W_context.weight[word2_idx]
    score1=cosine_similarity(w1_centre,w2_context)

    # consider context of word1 and centre of word2
    w1_context=network.W_context.weight[word1_idx]
    w2_centre=network.W_centre.weight[word2_idx]
    score2=cosine_similarity(w1_context,w2_centre)

    cross_role_score=max(score1,score2)
    centre_score=cosine_similarity(w1_centre,w2_centre)

    return centre_score, cross_role_score

In [25]:
corpus = [
    "the cat drinks water",
    "the dog drinks water",
    "the cat eats food",
    "the dog eats food",
    "The king is a man and she sits on a throne",
    "The queen is a woman and she sits on a throne",
    "The king is a man",
    "The queen is a woman"
]

data = word2vecDataset(window_size=2, k=5)
data.preprocess(corpus)
data.generate_training_pair()
data.prob_distribution()

network=SkipGramNS(data.vocab_size)
network.to(network.device)

print(f"Device: {network.device}")
history=network.fit(data, epochs=100, learning_rate=0.001)

c_score, s_score=similarity(data,network,"king","man")
print(f"Centre similarity score: {c_score}")
print(f"Cross similarity score: {s_score}")

Device: cuda
Epoch: 1 : Loss: 4.1542
Epoch: 2 : Loss: 4.0549
Epoch: 3 : Loss: 3.7306
Epoch: 4 : Loss: 3.3411
Epoch: 5 : Loss: 3.0435
Epoch: 6 : Loss: 2.8679
Epoch: 7 : Loss: 2.7901
Epoch: 8 : Loss: 2.7227
Epoch: 9 : Loss: 2.7020
Epoch: 10 : Loss: 2.6986
Epoch: 11 : Loss: 2.6811
Epoch: 12 : Loss: 2.6794
Epoch: 13 : Loss: 2.6634
Epoch: 14 : Loss: 2.6211
Epoch: 15 : Loss: 2.6344
Epoch: 16 : Loss: 2.6142
Epoch: 17 : Loss: 2.6101
Epoch: 18 : Loss: 2.5848
Epoch: 19 : Loss: 2.5527
Epoch: 20 : Loss: 2.5238
Epoch: 21 : Loss: 2.5114
Epoch: 22 : Loss: 2.4874
Epoch: 23 : Loss: 2.4616
Epoch: 24 : Loss: 2.4120
Epoch: 25 : Loss: 2.3763
Epoch: 26 : Loss: 2.3853
Epoch: 27 : Loss: 2.3325
Epoch: 28 : Loss: 2.3038
Epoch: 29 : Loss: 2.2687
Epoch: 30 : Loss: 2.2221
Epoch: 31 : Loss: 2.2234
Epoch: 32 : Loss: 2.1374
Epoch: 33 : Loss: 2.1427
Epoch: 34 : Loss: 2.0845
Epoch: 35 : Loss: 2.1004
Epoch: 36 : Loss: 2.0565
Epoch: 37 : Loss: 2.0855
Epoch: 38 : Loss: 2.0100
Epoch: 39 : Loss: 1.9955
Epoch: 40 : Loss: 2.0